In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from anthropic import Anthropic
import base64

client = Anthropic()
model = "claude-haiku-5-5"

In [9]:
from anthropic.types import Message 

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_efforts="medium"
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if thinking:
        params["thinking"] = {"type": "adaptive", "display": "summarized"}
        params["output_config"] = {"effort": thinking_effort}

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message 

def text_from_message(message):
        return "\n".join([block.text for block in message.content if block.type == "text"])

In [10]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [11]:
#TODO REad image data, feed this into Claude

In [15]:
with open("images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(messages, [
    {
       "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_011Ch1vfLW42fZsdDJECCDCq', container=None, content=[ThinkingBlock(signature='CAQSlBkKEAgTGAI4AUIIdGhpbmtpbmcSDJJy+bxDk6DR6RRpxhoMrvqVVSQmQSjotianIjB4x886R+TfgGYDA3UJHrIb8/tI6JXLFzvzvQbOoY3//BWMXW/lrQ75Jsii0TxIZaAqsRjnuCKETpYdqgwT/aJEccnmLty60rLBw+FG+dKg5rCMhBWh5GPww2+vCxEjwC5NVu1z/+cBV2/JWAorVwaZR37tN4zHI7Ge7/ERt4JsCjcv4ZiS/4rnKfCB34H2ta9NDy4k+abDshL2fL29XBYX8kHd+gihkMXfT4ookAgJDai7tZgT6cb86OzjOdzzysFNd55F5/avBL/TzAUdT7C1LoMilw4zw0KZZs7LsdjaF5X2FFdLdNHL57/1K0B+W6FNW4aAKyjpj8BpTzCgkDGmrlTyi0/egdSQyQycBMPEAE59oLQWVAYzKLo9ZS9RbjMNJoWH8/Oki8PBa+WMVEkuAqIKAHVidY7TbpxMDxjupBJfeTunYYamE9i1XnnqBDg/I7dXIQgxrJtygzcb3kJFSU0YNP6imIxXxZRVboC2i91vFC8DFdHDhQXjAc/3QbIiSU6HuHIWcdmQPNFTA1Sm3z467oS5zV9/KdVF9h46Z6PK2lV8X6SRAy81q9+Jy6kQFoT1IJXx541tZrkieTjxECh+orefZLAuPHV85MqV36Rc2qPMeo/QMPY08XG5uescdfU4Sj6VaYpMUYAHK/OI0l51+e0hh+PO1mJ/1gpt8WBONDr6qBHbHDLePcQyXD+vmcENL6iNEIbjapC7NV8qwH5aa+cvFLPTx6AsE9MV0w9mCbfSwyjJArW0OxeHcDvKP6cMYTakFafRK9CuBUR3vhYwuHt6ha15BPVBLnm3L89LX+1cjy1FvOhaZ2p0JW418Xn